> **QAC reproduction record** — claim `femoco-reiher-2017`, reproduction `qdk-trotter-2026-11`.
>
> This notebook is the executable evidence behind the QAC assessment of Reiher et al. (2017), *PNAS* **114**, 7555
> ([arXiv:1605.03590](https://arxiv.org/abs/1605.03590)). It is kept readable on purpose: the markdown walks through
> the method so it can be used for learning. The **last cell writes `results.json`**, the only source of the numbers
> the public record attributes to QAC; nothing in that file is typed by hand. The exact environment is pinned in
> `requirements.lock.txt`, and `run.json` records who executed it and when. Re-execute with `qacclaims execute`.

# FeMoCo by Trotter-based phase estimation

Reiher et al. (2017) gave the first detailed resource estimate for a chemistry problem of real interest: the
ground-state energy of FeMoCo, the active site of nitrogenase. Their algorithm is the simplest one there is.

1. Write the electronic Hamiltonian of an active space in second quantization and map it to qubits with the
   Jordan–Wigner transformation. It becomes a sum of Pauli strings, $H=\sum_{j=1}^{L}c_jP_j$.
2. Approximate $e^{-iHt}$ by a **second-order Trotter–Suzuki step**,
   $U(t)=\prod_{j=1}^{L}e^{-ic_jP_jt/2}\prod_{j=L}^{1}e^{-ic_jP_jt/2}$.
   Each exponential costs one arbitrary-angle rotation plus Clifford gates.
3. Run **phase estimation** on $U(t)$ to read off the energy. Resolving the energy to $\epsilon$ takes about
   $\alpha/(\epsilon t)$ applications of $U(t)$.
4. Compile every rotation into Clifford and T gates. T gates are the expensive ones under error correction, so the
   **T count** is the headline cost.

For structure 1 (54 electrons in 54 spatial orbitals, 108 spin orbitals) and a target of 0.1 mHa, the paper's
Table 1 reports about $1.1\times10^{15}$ T gates, 111 logical qubits, and 130 days at one logical T gate per 10 ns
(serial ordering). Table 2 adds a surface-code estimate: about $1.8\times10^{8}$ physical qubits at a physical error
rate of $10^{-3}$ and $1.2\times10^{6}$ at $10^{-6}$.

This notebook rebuilds those figures in five steps:

1. load the paper's own FeMoCo integrals;
2. count the Pauli terms of the Jordan–Wigner Hamiltonian, checking the count against QDK Chemistry;
3. count the rotations in one Trotter step, checking the rule against circuits compiled by QDK Chemistry;
4. apply the paper's cost model (its Eq. E2) to get Trotter steps, T gates, and logical runtime;
5. hand the logical counts to the QDK resource estimator for a physical estimate.

Locators such as "Eq. E2" and "Table IV" refer to the appendices of arXiv:1605.03590v2; Tables 1 and 2 are the
main-text tables of the PNAS version.

In [1]:
import hashlib
import itertools
import json
import math
import platform
import re
import tempfile
import urllib.request
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path

import numpy as np
import pandas as pd

from qdk_chemistry.utils import Logger

Logger.set_global_level(Logger.LogLevel.off)
pd.set_option("display.float_format", lambda value: f"{value:.4g}")

## What the paper states

Everything the reproduction takes from the paper is collected here, with its location. The first block is the
paper's cost model; the second is the set of figures we compare against at the end. The comparison figures are
used only for display in this notebook: the public record takes them from `estimates/reported.yaml`.

In [2]:
# Inputs of the paper's cost model (structure 1, serial ordering, "rescaled" Trotter number).
TARGET_PRECISION = 1e-4                  # Ha; total error budget of 0.1 mHa (Table 1; Eq. E1)
TROTTER_ORDER = 2                        # second-order Trotter-Suzuki (Strang) formula (Eq. C3)
PHASE_ESTIMATION_CONSTANT = math.pi / 2  # alpha: M = alpha / (eps * t) applications of the step (App. B; Table IV)
TROTTER_NUMBER = 166                     # beta: steps per 1/Ha for a 0.1 mHa Trotter error, "Rescaled" row (Table IV)
TROTTER_NUMBER_REFERENCE_ACCURACY = 1e-4 # Ha; the accuracy at which beta is defined (App. E)
SYNTHESIS_SLOPE = 1.15                   # gamma: T gates per rotation = gamma * log2(1/eps) + delta (Eq. A5; Table IV)
SYNTHESIS_OFFSET = 9.2                   # delta (Eq. A5; Table IV)
SYNTHESIS_ANCILLAS = 1                   # "This approach requires one additional ancilla qubit" (App. A.2)
PAPER_TERM_COUNT = 6.1e6                 # M, "for nitrogenase in the 54 orbital basis" (App. E)
LOGICAL_T_GATE_TIME_NS = 10              # "a logical T gate every 10 ns" (Sec. III; Table 1)
PHYSICAL_GATE_TIME_NS = 10               # "each physical gate takes 10 ns" (Sec. III; Table 2: 100 MHz)

# Figures to compare against (display only).
PAPER_FIGURES = {
    "T gates (Table 1)": 1.1e15,
    "T gates, additive error budget (Table IV, rescaled)": 1.2e15,
    "logical qubits (Table 1)": 111,
    "runtime at 10 ns per T gate, days (Table 1)": 130,
    "physical qubits at 1e-3 (Table 2)": 1.8e8,
    "physical qubits at 1e-6 (Table 2)": 1.2e6,
    "code distance at 1e-3 (Table 2)": 35,
    "code distance at 1e-6 (Table 2)": 9,
}

## Step 1: the Hamiltonian

The integrals are the original files of Reiher et al., published with the first author's permission in
[microsoft/qdk-chemistry-data](https://github.com/microsoft/qdk-chemistry-data/tree/main/data/hamiltonians/FeMoco-Reiher2017).
We fetch the structure-1 file (spin 0, charge +3, 54 electrons in 54 orbitals) from a pinned commit and check its
SHA-256 before using it. The file is a standard FCIDUMP: one-based indices, chemists' ordering $(pq|rs)$, and
eightfold permutational symmetry.

In [3]:
INTEGRALS_URL = (
    "https://raw.githubusercontent.com/microsoft/qdk-chemistry-data/"
    "963bf6d226b877fef31c7a856f3f705a46440295/data/hamiltonians/FeMoco-Reiher2017/"
    "FeMoco_S0_charge%2B3_orbsRAS24.16_targetCAS54.54_test3.fcidump"
)
INTEGRALS_SHA256 = "84f3a34d61cb1552797cc2574159ec4b6efa641c380631fb4f66be654a34940e"


def fetch_verified(url, sha256):
    # Download once into the system temp directory and verify the checksum on every use.
    cache = Path(tempfile.gettempdir()) / "qac-claims-cache"
    cache.mkdir(exist_ok=True)
    path = cache / f"{sha256[:16]}.fcidump"
    if not path.exists():
        urllib.request.urlretrieve(url, path)
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    if digest != sha256:
        path.unlink()
        raise RuntimeError(f"checksum mismatch for {url}: got {digest}")
    return path


def read_fcidump(path):
    # Return (orbitals, electrons, 2*S_z, core energy, h[p,q], (pq|rs)) from a restricted FCIDUMP file.
    with open(path, "r", encoding="ascii") as handle:
        header = []
        for line in handle:
            header.append(line)
            if "&END" in line.upper() or line.strip() == "/":
                break
        text = " ".join(header)
        norb = int(re.search(r"NORB\s*=\s*(\d+)", text).group(1))
        nelec = int(re.search(r"NELEC\s*=\s*(\d+)", text).group(1))
        ms2 = int(re.search(r"MS2\s*=\s*(-?\d+)", text).group(1))
        records = np.loadtxt(handle, dtype=np.float64)
    values, index = records[:, 0], records[:, 1:].astype(np.int64)
    two_body = index[:, 2] > 0
    one_body = ~two_body & (index[:, 0] > 0)
    core_energy = float(values[index[:, 0] == 0].sum())
    h = np.zeros((norb, norb))
    i, j = index[one_body, 0] - 1, index[one_body, 1] - 1
    h[i, j] = h[j, i] = values[one_body]
    eri = np.zeros((norb,) * 4)
    i, j, k, l = (index[two_body, column] - 1 for column in range(4))
    for a, b, c, d in ((i, j, k, l), (j, i, k, l), (i, j, l, k), (j, i, l, k), (k, l, i, j), (l, k, i, j), (k, l, j, i), (l, k, j, i)):
        eri[a, b, c, d] = values[two_body]
    return norb, nelec, ms2, core_energy, h, eri


NUM_ORBITALS, NUM_ELECTRONS, MS2, CORE_ENERGY, ONE_BODY, TWO_BODY = read_fcidump(fetch_verified(INTEGRALS_URL, INTEGRALS_SHA256))
NUM_SPIN_ORBITALS = 2 * NUM_ORBITALS
print(f"{NUM_ELECTRONS} electrons in {NUM_ORBITALS} spatial orbitals ({NUM_SPIN_ORBITALS} spin orbitals), 2S_z = {MS2}")
print(f"core energy {CORE_ENERGY:.6f} Ha")
print(f"smallest |(pq|rs)| = {np.abs(TWO_BODY).min():.3e}: no two-electron integral vanishes")

54 electrons in 54 spatial orbitals (108 spin orbitals), 2S_z = 0
core energy -13212.970326 Ha
smallest |(pq|rs)| = 3.921e-11: no two-electron integral vanishes


## Step 2: Pauli terms of the Jordan–Wigner Hamiltonian

The cost of a Trotter step is set by the number $L$ of Pauli strings in

$$
H=\sum_{pq,\sigma}h_{pq}\,a^\dagger_{p\sigma}a_{q\sigma}
+\tfrac12\sum_{pqrs,\sigma\tau}(pq|rs)\,a^\dagger_{p\sigma}a^\dagger_{r\tau}a_{s\tau}a_{q\sigma}
$$

after the Jordan–Wigner mapping. QDK Chemistry's `qdk` qubit mapper performs that mapping, but it returns one
Python string per Pauli term, and at 54 orbitals there are more than ten million 108-character strings. So we do
two things.

**A closed-form count.** For real integrals the Pauli strings fall into a few classes whose coefficients can be
written down directly. With $J_{pq}=(pp|qq)$, $K_{pq}=(pq|qp)$, and $n=(1-Z)/2$:

| Class | Operator | Pauli strings | $|$coefficient$|$ |
| --- | --- | --- | --- |
| number | $n_{p\sigma}$ | $Z$ | $\left|\tfrac12h_{pp}+\tfrac14\sum_{q\tau\neq p\sigma}(J_{pq}-\delta_{\sigma\tau}K_{pq})\right|$ |
| number–number | $n_{p\sigma}n_{q\tau}$ | $ZZ$ | $\tfrac14\left|J_{pq}-\delta_{\sigma\tau}K_{pq}\right|$ |
| hopping | $a^\dagger_{p\sigma}a_{q\sigma}+\text{h.c.}$ | $XZ{\cdots}ZX$, $YZ{\cdots}ZY$ | $\tfrac12\left|h_{pq}+\tfrac12\sum_{k\tau}g^{pq}_{k\tau}\right|$ |
| hopping × number | $(a^\dagger_{p\sigma}a_{q\sigma}+\text{h.c.})\,n_{k\tau}$ | the two above, times $Z_{k\tau}$ | $\tfrac14\left|g^{pq}_{k\tau}\right|$, with $g^{pq}_{k\tau}=(pq|kk)-\delta_{\sigma\tau}(pk|kq)$ |
| double excitation, mixed spin | $p{<}q$ of spin $\alpha$, $r{<}s$ of spin $\beta$ | 4 | $\tfrac14\left|(pq|rs)\right|$ |
| double excitation, same spin | $p{<}q{<}r{<}s$ | 6 | $\tfrac14|A-B|,\ \tfrac14|A-C|,\ \tfrac14|B-C|$, twice each |

where $A=(pq|rs)$, $B=(pr|qs)$, $C=(ps|qr)$. A same-spin quadruple gives six strings, not eight, because the three
excitation amplitudes $B-C$, $A-C$, $A-B$ are linearly dependent.

**A check against QDK Chemistry.** On truncated active spaces (the first $n$ orbitals of the FeMoCo integrals) we
run the QDK Chemistry mapper and require that it returns exactly the same multiset of coefficient magnitudes as the
table above. The closed form is then applied to all 54 orbitals with the mapper's own default threshold.

In [4]:
PAULI_COEFFICIENT_THRESHOLD = 1e-12  # the QDK Chemistry qubit mapper's default pruning threshold


def jordan_wigner_classes(h, eri):
    # |coefficient| of every non-identity Pauli string of the Jordan-Wigner Hamiltonian, by class.
    # Returns {class name: (magnitudes, number of Pauli strings that carry each magnitude)}.
    n = h.shape[0]
    coulomb = np.einsum("ppqq->pq", eri)
    exchange = np.einsum("pqqp->pq", eri)
    off_diagonal = ~np.eye(n, dtype=bool)
    z = 0.5 * np.diag(h) + 0.25 * (((coulomb - exchange) * off_diagonal).sum(axis=1) + coulomb.sum(axis=1))
    p, q = np.triu_indices(n, k=1)
    g_opposite = np.einsum("pqkk->pqk", eri)[p, q, :]
    g_same = g_opposite - np.einsum("pkkq->pqk", eri)[p, q, :]
    other = np.ones_like(g_same, dtype=bool)   # k different from p and q, for the same-spin sum
    pairs = np.arange(len(p))
    other[pairs, p] = other[pairs, q] = False
    hopping = 0.5 * (h[p, q] + 0.5 * ((g_same * other).sum(axis=1) + g_opposite.sum(axis=1)))
    mixed = 0.25 * eri[p[:, None], q[:, None], p[None, :], q[None, :]].ravel()
    if n >= 4:
        a, b, c, d = np.array(list(itertools.combinations(range(n), 4)), dtype=np.int64).T
        A, B, C = eri[a, b, c, d], eri[a, c, b, d], eri[a, d, b, c]
        same = 0.25 * np.concatenate([A - B, A - C, B - C])
    else:
        same = np.zeros(0)
    return {
        "number (Z)": (np.abs(z), 2),
        "number-number (ZZ), same spin": (0.25 * np.abs((coulomb - exchange)[p, q]), 2),
        "number-number (ZZ), opposite spin": (0.25 * np.abs(coulomb).ravel(), 1),
        "hopping": (np.abs(hopping), 4),
        "hopping x number, same spin": (0.25 * np.abs(g_same[other]), 4),
        "hopping x number, opposite spin": (0.25 * np.abs(g_opposite).ravel(), 4),
        "double excitation, mixed spin": (np.abs(mixed), 4),
        "double excitation, same spin": (np.abs(same), 4),
    }


def kept_magnitudes(classes, threshold):
    magnitudes = np.concatenate([np.repeat(values, copies) for values, copies in classes.values()])
    return np.sort(magnitudes[magnitudes > threshold])

In [5]:
from qdk_chemistry.algorithms import create
from qdk_chemistry.data import CanonicalFourCenterHamiltonianContainer, Hamiltonian, MajoranaMapping, ModelOrbitals

VALIDATION_SIZES = (2, 4, 6, 8)  # spatial orbitals kept in each truncated active space


def truncated_qubit_hamiltonian(n):
    # Jordan-Wigner qubit Hamiltonian of the first n FeMoCo orbitals, built by QDK Chemistry.
    hamiltonian = Hamiltonian(
        CanonicalFourCenterHamiltonianContainer(
            one_body_integrals=ONE_BODY[:n, :n].copy(),
            two_body_integrals=TWO_BODY[:n, :n, :n, :n].copy().ravel(),
            orbitals=ModelOrbitals(n),
            core_energy=0.0,
            inactive_fock_matrix=np.zeros((n, n)),
        )
    )
    return create("qubit_mapper", "qdk", threshold=PAULI_COEFFICIENT_THRESHOLD).run(hamiltonian, MajoranaMapping.jordan_wigner(2 * n))


validation_rows = []
VALIDATION_OPERATORS = {}
for n in VALIDATION_SIZES:
    operator = truncated_qubit_hamiltonian(n)
    VALIDATION_OPERATORS[n] = operator
    is_identity = np.array([set(label) == {"I"} for label in operator.pauli_strings])
    from_qdk = np.sort(np.abs(np.asarray(operator.coefficients)[~is_identity]))
    from_closed_form = kept_magnitudes(jordan_wigner_classes(ONE_BODY[:n, :n], TWO_BODY[:n, :n, :n, :n]), PAULI_COEFFICIENT_THRESHOLD)
    assert from_qdk.shape == from_closed_form.shape, f"term counts differ for {n} orbitals"
    assert np.allclose(from_qdk, from_closed_form, rtol=1e-9, atol=1e-13), f"coefficients differ for {n} orbitals"
    validation_rows.append({"orbitals": n, "qubits": 2 * n, "Pauli terms (QDK Chemistry)": len(from_qdk), "Pauli terms (closed form)": len(from_closed_form), "largest coefficient difference": float(np.abs(from_qdk - from_closed_form).max())})

print("QDK Chemistry and the closed form agree term for term on every truncated active space:")
display(pd.DataFrame(validation_rows))

QDK Chemistry and the closed form agree term for term on every truncated active space:


,orbitals,qubits,Pauli terms (QDK Chemistry),Pauli terms (closed form),largest coefficient difference
0,2,4,26,26,3.469e-18
1,4,8,360,360,1.776e-15
2,6,12,1818,1818,1.776e-15
3,8,16,5792,5792,1.776e-15


In [6]:
FEMOCO_CLASSES = jordan_wigner_classes(ONE_BODY, TWO_BODY)
class_rows = [
    {"class": name, "Pauli strings": int((values > PAULI_COEFFICIENT_THRESHOLD).sum()) * copies, "smallest |coefficient|": float(values.min()), "largest |coefficient|": float(values.max())}
    for name, (values, copies) in FEMOCO_CLASSES.items()
]
display(pd.DataFrame(class_rows))

FEMOCO_MAGNITUDES = kept_magnitudes(FEMOCO_CLASSES, PAULI_COEFFICIENT_THRESHOLD)
NUM_PAULI_TERMS = int(FEMOCO_MAGNITUDES.size)
ONE_NORM = float(FEMOCO_MAGNITUDES.sum())
print(f"Non-identity Pauli terms L = {NUM_PAULI_TERMS:,}")
print(f"The paper quotes M = {PAPER_TERM_COUNT:.3g} (App. E); L / M = {NUM_PAULI_TERMS / PAPER_TERM_COUNT:.3f}")
print(f"Sum of |coefficients| = {ONE_NORM:.4f} Ha")

,class,Pauli strings,smallest |coefficient|,largest |coefficient|
0,number (Z),108,0.07448,1.355
1,"number-number (ZZ), same spin",2862,0.01279,0.1197
2,"number-number (ZZ), opposite spin",2916,0.01279,0.1607
3,hopping,5724,9.305e-08,0.1747
4,"hopping x number, same spin",297648,5.128e-09,0.05372
5,"hopping x number, opposite spin",309096,5.125e-10,0.05607
6,"double excitation, mixed spin",8191044,9.804e-12,0.05035
7,"double excitation, same spin",3795012,4.592e-11,0.0215


Non-identity Pauli terms L = 12,604,410
The paper quotes M = 6.1e+06 (App. E); L / M = 2.066
Sum of |coefficients| = 1863.2501 Ha


The 54-orbital Hamiltonian has about twice as many Pauli terms as the $M=6.1\times10^{6}$ the paper quotes. We
could not determine how the paper arrived at its value. Dropping small terms does not explain it: the table below
shows that the count barely moves until the threshold is large enough to discard a chemically significant part of
the Hamiltonian. One reading that would give a number close to the paper's is a count of fermionic terms, in which
each pair of Pauli strings of the form $XZ{\cdots}ZX+YZ{\cdots}ZY$ is counted once. In the paper's formula $M$ is
the number of single-qubit rotations per sweep, so we use the Pauli-term count.

In [7]:
all_magnitudes = kept_magnitudes(FEMOCO_CLASSES, 0.0)
display(pd.DataFrame([
    {"threshold (Ha)": threshold, "Pauli terms kept": int((all_magnitudes > threshold).sum()), "discarded sum of |coefficients| (Ha)": float(all_magnitudes[all_magnitudes <= threshold].sum())}
    for threshold in (1e-12, 1e-10, 1e-8, 1e-6, 1e-5, 1e-4)
]))

,threshold (Ha),Pauli terms kept,discarded sum of |coefficients| (Ha)
0,1e-12,12604410,0
1,1e-10,12604334,4.173e-09
2,1e-08,12599502,2.508e-05
3,1e-06,12186930,0.2031
4,1e-05,9745842,12.48
5,0.0001,3283430,273.5


## Step 3: rotations in one Trotter step

QDK Chemistry's `trotter` builder turns a Pauli-decomposed Hamiltonian into a product formula, and its
`pauli_sequence` circuit mapper turns the formula into a circuit: each exponential $e^{-i\theta P}$ becomes a basis
change, a CNOT ladder, and **one** $R_z$ rotation. A second-order step sweeps through the $L$ terms forwards and
backwards, and the two exponentials of the last term merge, so one step should contain $2L-1$ rotations on $2n$
qubits.

The cell below checks that rule. For each truncated active space it builds the second-order formula with QDK
Chemistry, compiles the circuit, and counts its logical resources with the QDK resource estimator.

Phase estimation needs the step to be controlled by one extra qubit. QDK Chemistry's stock controlled mapper
controls each rotation in the standard way, which doubles the rotation count; the last column shows this. The paper
avoids the doubling (its Lemma 2 and Fig. 6): a CNOT from the control qubit on each side of a rotation flips the
sign of the angle, so the control qubit selects between $U$ and $U^\dagger$ using Clifford gates only. We follow the
paper: one rotation per exponential, plus one control qubit.

In [8]:
CHECK_REPETITIONS = 3  # Trotter steps compiled in each check circuit

step_rows = []
for n in VALIDATION_SIZES:
    operator = VALIDATION_OPERATORS[n]
    terms = len(operator.coefficients) - 1  # non-identity Pauli terms
    formula = create("hamiltonian_unitary_builder", "trotter", order=TROTTER_ORDER, time=0.01, num_divisions=CHECK_REPETITIONS).run(operator)
    plain = create("circuit_mapper", "pauli_sequence").run(formula).estimate().logical_counts
    controlled = create("controlled_circuit_mapper", "pauli_sequence").run(formula).estimate().logical_counts
    assert plain["rotationCount"] == CHECK_REPETITIONS * (2 * terms - 1), f"unexpected rotation count for {n} orbitals"
    assert plain["numQubits"] == 2 * n and controlled["numQubits"] == 2 * n + 1
    assert plain["tCount"] == plain["cczCount"] == plain["ccixCount"] == 0
    step_rows.append({
        "orbitals": n,
        "Pauli terms L": terms,
        "rotations per step (compiled)": plain["rotationCount"] // CHECK_REPETITIONS,
        "2L - 1": 2 * terms - 1,
        "qubits": plain["numQubits"],
        "rotations per step, stock controlled mapper": controlled["rotationCount"] // CHECK_REPETITIONS,
        "qubits, controlled": controlled["numQubits"],
    })

display(pd.DataFrame(step_rows))

,orbitals,Pauli terms L,rotations per step (compiled),2L - 1,qubits,"rotations per step, stock controlled mapper","qubits, controlled"
0,2,26,51,51,4,104,5
1,4,360,719,719,8,1440,9
2,6,1818,3635,3635,12,7272,13
3,8,5792,11583,11583,16,23168,17


In [9]:
ROTATIONS_PER_STEP = 2 * NUM_PAULI_TERMS - 1
PHASE_ESTIMATION_QUBITS = 1
ALGORITHM_QUBITS = NUM_SPIN_ORBITALS + PHASE_ESTIMATION_QUBITS   # what the Trotter circuit itself uses
LOGICAL_QUBITS = ALGORITHM_QUBITS + SYNTHESIS_ANCILLAS           # plus the paper's rotation-synthesis ancilla

print(f"Rotations in one second-order Trotter step of FeMoCo: {ROTATIONS_PER_STEP:,}")
print(f"The paper's formula uses 2M = {2 * PAPER_TERM_COUNT:.3g}")
print(f"Logical qubits: {NUM_SPIN_ORBITALS} spin orbitals + {PHASE_ESTIMATION_QUBITS} phase-estimation qubit + {SYNTHESIS_ANCILLAS} synthesis ancilla = {LOGICAL_QUBITS}")

Rotations in one second-order Trotter step of FeMoCo: 25,208,819
The paper's formula uses 2M = 1.22e+07
Logical qubits: 108 spin orbitals + 1 phase-estimation qubit + 1 synthesis ancilla = 110


The full 54-orbital circuit is not compiled here. Its second-order step has tens of millions of exponentials, and
building them as Python objects needs more memory than a laptop has. The rule $2L-1$ is exact for every circuit we
did compile, and it depends only on $L$.

## Step 4: the paper's cost model

The paper's Appendix E writes the T count as

$$
C=\underbrace{2M}_{\text{rotations per step}}\times
\underbrace{\frac{\alpha}{\epsilon_1}\,\beta\sqrt{\frac{\epsilon}{\epsilon_2}}}_{\text{Trotter steps}}\times
\underbrace{\left[\gamma\log_2\!\left(\frac{2M}{\epsilon_3}\,\beta\sqrt{\frac{\epsilon}{\epsilon_2}}\right)+\delta\right]}_{\text{T gates per rotation}}
\qquad\text{(Eq. E2)}
$$

and reads as follows.

- **Trotter error** $\epsilon_2$. The Trotter number $\beta$ is the number of steps per unit time (1/Ha) at which the
  second-order error in the energy is $\epsilon=0.1$ mHa. The error grows as $t^2$, so the step is
  $t=\beta^{-1}\sqrt{\epsilon_2/\epsilon}$.
- **Phase-estimation error** $\epsilon_1$. Resolving the energy to $\epsilon_1$ takes $\alpha/(\epsilon_1t)$
  applications of the step, with $\alpha=\pi/2$.
- **Synthesis error** $\epsilon_3$. Each of the rotations in a step may be off by $\epsilon_3t$ divided by the
  number of rotations, and a rotation of accuracy $\Delta$ costs $\gamma\log_2(1/\Delta)+\delta$ T gates on average.

The three errors share the 0.1 mHa budget. The main text uses $\epsilon_2+\sqrt{\epsilon_1^2+\epsilon_3^2}=\epsilon$
(Appendix E.1); Table IV uses the plain sum. The paper chooses the split by minimizing a simplified cost (Eq. E3,
which drops $\sqrt{\epsilon/\epsilon_2}$ from the logarithm) and then evaluates Eq. E2 at that split. We do the same.

The Trotter number is the one input we cannot rebuild. The paper obtains $\beta=166$ by dividing a rigorous bound
for FeMoCo by the average factor by which the same bound overestimates the Trotter error of small molecules it
could simulate exactly. We take $\beta$ from Table IV.

The split is found on a fixed grid, and the T gates per rotation are rounded to three decimals, so that the result
does not depend on the platform's floating-point library.

In [10]:
BUDGET_GRID = np.arange(1, 1000) / 1000.0  # candidate fractions of the error budget, in steps of 0.001


def cost_model(rotations_per_step, rule):
    # Evaluate the paper's Eq. E2 at the error split that minimizes its Eq. E3.
    # rule = "variance": eps_2 + sqrt(eps_1^2 + eps_3^2) = eps (main text); rule = "sum": eps_1 + eps_2 + eps_3 = eps (Table IV).
    trotter, synthesis = np.meshgrid(BUDGET_GRID, BUDGET_GRID, indexing="ij")
    with np.errstate(invalid="ignore"):
        phase = np.sqrt((1 - trotter) ** 2 - synthesis**2) if rule == "variance" else 1 - trotter - synthesis
    feasible = np.nan_to_num(phase, nan=0.0) > 1e-9
    phase = np.where(feasible, phase, 1.0)
    simplified = (1 / (phase * np.sqrt(trotter))) * (
        SYNTHESIS_SLOPE * np.log2(rotations_per_step * TROTTER_NUMBER / (synthesis * TARGET_PRECISION)) + SYNTHESIS_OFFSET
    )
    simplified = np.where(feasible, simplified, np.inf)
    best = np.unravel_index(np.argmin(simplified), simplified.shape)
    f_trotter, f_synthesis = float(BUDGET_GRID[best[0]]), float(BUDGET_GRID[best[1]])
    f_phase = math.sqrt((1 - f_trotter) ** 2 - f_synthesis**2) if rule == "variance" else 1 - f_trotter - f_synthesis
    eps_phase, eps_trotter, eps_synthesis = (f * TARGET_PRECISION for f in (f_phase, f_trotter, f_synthesis))

    steps_per_unit_time = TROTTER_NUMBER * math.sqrt(TROTTER_NUMBER_REFERENCE_ACCURACY / eps_trotter)
    trotter_steps = math.ceil(PHASE_ESTIMATION_CONSTANT / eps_phase * steps_per_unit_time)
    t_per_rotation = round(SYNTHESIS_SLOPE * math.log2(rotations_per_step * steps_per_unit_time / eps_synthesis) + SYNTHESIS_OFFSET, 3)
    rotations = trotter_steps * rotations_per_step
    t_count = (rotations * round(t_per_rotation * 1000) + 500) // 1000  # integer arithmetic, exact
    return {
        "eps_phase": eps_phase,
        "eps_trotter": eps_trotter,
        "eps_synthesis": eps_synthesis,
        "trotter_steps": trotter_steps,
        "rotations": rotations,
        "t_per_rotation": t_per_rotation,
        "t_count": t_count,
        "runtime_seconds": t_count * LOGICAL_T_GATE_TIME_NS / 1e9,
    }


ERROR_BUDGET_RULE = "variance"
MODEL = cost_model(ROTATIONS_PER_STEP, ERROR_BUDGET_RULE)
TROTTER_STEPS = MODEL["trotter_steps"]
ROTATION_COUNT = MODEL["rotations"]
T_PER_ROTATION = MODEL["t_per_rotation"]
T_COUNT = MODEL["t_count"]
LOGICAL_RUNTIME_SECONDS = MODEL["runtime_seconds"]

print(f"Error split (mHa): phase estimation {MODEL['eps_phase'] * 1e3:.4f}, Trotter {MODEL['eps_trotter'] * 1e3:.4f}, synthesis {MODEL['eps_synthesis'] * 1e3:.4f}")
print(f"Trotter steps:        {TROTTER_STEPS:,}")
print(f"Rotations:            {ROTATION_COUNT:.4g}")
print(f"T gates per rotation: {T_PER_ROTATION}")
print(f"T gates:              {T_COUNT:.4g}   (paper, Table 1: {PAPER_FIGURES['T gates (Table 1)']:.2g})")
print(f"Runtime at {LOGICAL_T_GATE_TIME_NS} ns per T gate: {LOGICAL_RUNTIME_SECONDS / 86400:,.0f} days   (paper, Table 1: {PAPER_FIGURES['runtime at 10 ns per T gate, days (Table 1)']} days)")

Error split (mHa): phase estimation 0.0664, Trotter 0.0328, synthesis 0.0106
Trotter steps:        6,861,089
Rotations:            1.73e+14
T gates per rotation: 65.886
T gates:              1.14e+16   (paper, Table 1: 1.1e+15)
Runtime at 10 ns per T gate: 1,319 days   (paper, Table 1: 130 days)


### Where the difference comes from

Our T count is an order of magnitude above the paper's. Two separate factors make up the gap, and the cell below
measures each.

1. **Rotations per step.** We count $2L-1$ rotations where the paper's formula uses $2M$ (Step 2).
2. **The formula against the table.** Putting the paper's own $M=6.1\times10^{6}$ into Eq. E2, with the constants
   of Table IV, does not return the paper's T count; it returns several times more. A similar factor appears for
   the plain-sum budget of Table IV, so it does not come from the choice of budget rule.

We did not find the cause of the second factor, and we did not adjust any input to remove it.

In [11]:
PAPER_MODEL = cost_model(round(2 * PAPER_TERM_COUNT), "variance")
PAPER_MODEL_SUM = cost_model(round(2 * PAPER_TERM_COUNT), "sum")

display(pd.DataFrame([
    {"calculation": "paper, Table 1 (main text)", "rotations per step": 2 * PAPER_TERM_COUNT, "Trotter steps": float("nan"), "T gates per rotation": float("nan"), "T gates": PAPER_FIGURES["T gates (Table 1)"]},
    {"calculation": "paper, Table IV (rescaled, plain-sum budget)", "rotations per step": 2 * PAPER_TERM_COUNT, "Trotter steps": float("nan"), "T gates per rotation": float("nan"), "T gates": PAPER_FIGURES["T gates, additive error budget (Table IV, rescaled)"]},
    {"calculation": "Eq. E2 with the paper's M, variance budget", "rotations per step": 2 * PAPER_TERM_COUNT, "Trotter steps": PAPER_MODEL["trotter_steps"], "T gates per rotation": PAPER_MODEL["t_per_rotation"], "T gates": PAPER_MODEL["t_count"]},
    {"calculation": "Eq. E2 with the paper's M, plain-sum budget", "rotations per step": 2 * PAPER_TERM_COUNT, "Trotter steps": PAPER_MODEL_SUM["trotter_steps"], "T gates per rotation": PAPER_MODEL_SUM["t_per_rotation"], "T gates": PAPER_MODEL_SUM["t_count"]},
    {"calculation": "this reproduction: Eq. E2 with our Pauli-term count", "rotations per step": ROTATIONS_PER_STEP, "Trotter steps": TROTTER_STEPS, "T gates per rotation": T_PER_ROTATION, "T gates": T_COUNT},
]))

print(f"Factor from rotations per step:        {T_COUNT / PAPER_MODEL['t_count']:.2f}")
print(f"Factor between Eq. E2 and Table 1:     {PAPER_MODEL['t_count'] / PAPER_FIGURES['T gates (Table 1)']:.2f}")
print(f"Factor between Eq. E2 and Table IV:    {PAPER_MODEL_SUM['t_count'] / PAPER_FIGURES['T gates, additive error budget (Table IV, rescaled)']:.2f}")
print(f"Overall, this reproduction / Table 1:  {T_COUNT / PAPER_FIGURES['T gates (Table 1)']:.2f}")

,calculation,rotations per step,Trotter steps,T gates per rotation,T gates
0,"paper, Table 1 (main text)",1.22e+07,NaN,NaN,1.1e+15
1,"paper, Table IV (rescaled, plain-sum budget)",1.22e+07,NaN,NaN,1.2e+15
2,"Eq. E2 with the paper's M, variance budget",1.22e+07,6.863e+06,64.67,5.414e+15
3,"Eq. E2 with the paper's M, plain-sum budget",1.22e+07,6.94e+06,67.82,5.742e+15
4,this reproduction: Eq. E2 with our Pauli-term ...,2.521e+07,6.861e+06,65.89,1.14e+16


Factor from rotations per step:        2.10
Factor between Eq. E2 and Table 1:     4.92
Factor between Eq. E2 and Table IV:    4.79
Overall, this reproduction / Table 1:  10.36


## Step 5: physical resources

The paper's Table 2 assumes a surface code with physical gates at 100 MHz, 15-to-1 magic-state distillation, and
enough T factories to deliver one logical T gate every 10 ns, so that the machine finishes in the 130 days of
Table 1. It follows a costing model from the literature of the time (its Ref. 48).

We use the [QDK resource estimator](https://github.com/microsoft/qdk/tree/main/samples/qre) (QRE) with the
assumptions the paper states and QRE's own model for everything else:

- `GateBased(error_rate, gate_time=10, measurement_time=10)`: physical gates and measurements of 10 ns, at an error
  rate of $10^{-3}$ or $10^{-6}$. The paper does not give a measurement time; we take it equal to the gate time.
- `SurfaceCode`: the rotated surface code. QRE chooses the distance.
- `RoundBasedFactory`: T factories built from rounds of 15-to-1 distillation, the scheme the paper names.
- `PSSPC * LatticeSurgery`: rotations are synthesized into T states and the circuit is laid out for lattice
  surgery. In this model a logical operation lasts one logical cycle, that is, a code distance's worth of syndrome
  rounds. It cannot go faster than that, so the paper's rate of one T gate per 10 ns is out of reach.
- A total failure probability of 1% for the run. The paper does not state one.

Two practical points.

**More T states per rotation.** QRE's rotation synthesis searches 5 to 20 T states per rotation by default. This
circuit has so many rotations that no setting in that range meets the error budget. We let it search up to 80.

**One Trotter step at a time.** QRE stores a runtime as a 64-bit number of nanoseconds, which overflows after about
585 years, and this computation is that long. So QRE estimates a single Trotter step with the failure budget
divided by the number of steps, and we multiply the runtime by the number of steps. Errors add linearly in QRE's
model and the factories are sized by a rate, so this gives the same machine as estimating the whole run; we
checked that on a shorter run where both fit.

QRE returns a set of trade-offs between qubits and runtime. We take the one with the smallest product of the two.

In [12]:
from qdk.estimator import LogicalCounts
from qdk.qre import PSSPC, LatticeSurgery, estimate, property_name
from qdk.qre.application import QSharpApplication
from qdk.qre.instruction_ids import LATTICE_SURGERY
from qdk.qre.models import GateBased, RoundBasedFactory, SurfaceCode

ERROR_BUDGET = 0.01                           # total failure probability allowed for the whole run
MEASUREMENT_TIME_NS = PHYSICAL_GATE_TIME_NS   # not stated in the paper; taken equal to the gate time
CODE_DISTANCES = list(range(3, 62, 2))
T_STATES_PER_ROTATION = list(range(5, 81))

# One Trotter step as the resource estimator sees it: serial rotations on the algorithm's qubits.
STEP_COUNTS = LogicalCounts({
    "numQubits": ALGORITHM_QUBITS,
    "tCount": 0,
    "rotationCount": ROTATIONS_PER_STEP,
    "rotationDepth": ROTATIONS_PER_STEP,
    "cczCount": 0,
    "ccixCount": 0,
    "measurementCount": 0,
})


def frontier(counts, physical_error_rate, max_error):
    # All qubit/runtime trade-offs QRE returns for the given logical counts.
    architecture = GateBased(error_rate=physical_error_rate, gate_time=PHYSICAL_GATE_TIME_NS, measurement_time=MEASUREMENT_TIME_NS)
    isa_query = SurfaceCode.q(distance=CODE_DISTANCES) * RoundBasedFactory.q(use_cache=False, code_query=SurfaceCode.q(distance=CODE_DISTANCES))
    trace_query = PSSPC.q(num_ts_per_rotation=T_STATES_PER_ROTATION) * LatticeSurgery.q()
    points = []
    for entry in estimate(QSharpApplication(counts), architecture, isa_query, trace_query, max_error=max_error):
        properties = {property_name(key): value for key, value in entry.properties.items()}
        code = entry.source[LATTICE_SURGERY].transform
        (factory,) = entry.factories.values()
        points.append({
            "physical_qubits": int(entry.qubits),
            "runtime_ns": int(entry.runtime),
            "error": float(entry.error),
            "distance": int(code.distance),
            "cycle_time_ns": code.one_qubit_gate_depth * PHYSICAL_GATE_TIME_NS + code.two_qubit_gate_depth * PHYSICAL_GATE_TIME_NS + MEASUREMENT_TIME_NS,
            "t_states_per_rotation": int(properties["NUM_TS_PER_ROTATION"]),
            "factory_copies": int(factory.copies),
            "compute_qubits": int(properties["PHYSICAL_COMPUTE_QUBITS"]),
            "factory_qubits": int(properties["PHYSICAL_FACTORY_QUBITS"]),
            "memory_qubits": int(properties.get("PHYSICAL_MEMORY_QUBITS", 0)),
        })
    return points


def smallest_volume(points):
    return min(points, key=lambda point: (point["physical_qubits"] * point["runtime_ns"], point["physical_qubits"]))


def physical_estimate(physical_error_rate):
    points = frontier(STEP_COUNTS, physical_error_rate, ERROR_BUDGET / TROTTER_STEPS)
    assert points, "the resource estimator found no machine within the error budget"
    best = dict(smallest_volume(points))
    best.update(
        physical_error_rate=physical_error_rate,
        frontier_points=len(points),
        runtime_seconds=best["runtime_ns"] * TROTTER_STEPS / 1e9,
        achieved_error=best["error"] * TROTTER_STEPS,
    )
    assert best["achieved_error"] <= ERROR_BUDGET
    return best


# Check the step-at-a-time shortcut on a run short enough for QRE to hold: 1,000 steps at once against 1 step scaled up.
CHECK_STEPS = 1000
whole_run = LogicalCounts({**dict(STEP_COUNTS), "rotationCount": ROTATIONS_PER_STEP * CHECK_STEPS, "rotationDepth": ROTATIONS_PER_STEP * CHECK_STEPS})
for rate in (1e-3, 1e-6):
    at_once = smallest_volume(frontier(whole_run, rate, ERROR_BUDGET * CHECK_STEPS / TROTTER_STEPS))
    one_step = smallest_volume(frontier(STEP_COUNTS, rate, ERROR_BUDGET / TROTTER_STEPS))
    assert at_once["physical_qubits"] == one_step["physical_qubits"] and at_once["distance"] == one_step["distance"]
    assert at_once["runtime_ns"] == one_step["runtime_ns"] * CHECK_STEPS
print("Estimating one step and scaling gives the same machine as estimating the whole run.")

ESTIMATE_1E3 = physical_estimate(1e-3)
ESTIMATE_1E6 = physical_estimate(1e-6)

SECONDS_PER_YEAR = 365.25 * 86400
display(pd.DataFrame([
    {
        "physical error rate": estimate_["physical_error_rate"],
        "physical qubits": estimate_["physical_qubits"],
        "runtime (years)": estimate_["runtime_seconds"] / SECONDS_PER_YEAR,
        "code distance": estimate_["distance"],
        "T factories": estimate_["factory_copies"],
        "T states per rotation": estimate_["t_states_per_rotation"],
        "qubits in factories": estimate_["factory_qubits"],
        "failure probability": estimate_["achieved_error"],
    }
    for estimate_ in (ESTIMATE_1E3, ESTIMATE_1E6)
]))

Estimating one step and scaling gives the same machine as estimating the whole run.


,physical error rate,physical qubits,runtime (years),code distance,T factories,T states per rotation,qubits in factories,failure probability
0,0.001,2721009,448.9,39,15,34,1963800,0.009708
1,1e-06,181529,103.6,9,17,34,141440,0.005329


## Comparison with the paper

The physical estimates differ from Table 2 in both directions, and for one reason. The paper's machine is built to
consume a T gate every 10 ns, which takes hundreds of T factories and makes them the bulk of its qubits. In the
lattice-surgery model a T gate takes a logical cycle, more than a hundred times longer, so far fewer factories keep
up, the machine is much smaller, and the run is much longer.

In [13]:
def ratio_row(figure, ours, paper):
    return {"figure": figure, "paper": paper, "this reproduction": ours, "ours / paper": ours / paper}


display(pd.DataFrame([
    ratio_row("logical qubits", LOGICAL_QUBITS, PAPER_FIGURES["logical qubits (Table 1)"]),
    ratio_row("T gates", T_COUNT, PAPER_FIGURES["T gates (Table 1)"]),
    ratio_row("runtime at 10 ns per T gate (days)", LOGICAL_RUNTIME_SECONDS / 86400, PAPER_FIGURES["runtime at 10 ns per T gate, days (Table 1)"]),
    ratio_row("physical qubits, error rate 1e-3", ESTIMATE_1E3["physical_qubits"], PAPER_FIGURES["physical qubits at 1e-3 (Table 2)"]),
    ratio_row("physical qubits, error rate 1e-6", ESTIMATE_1E6["physical_qubits"], PAPER_FIGURES["physical qubits at 1e-6 (Table 2)"]),
    ratio_row("code distance, error rate 1e-3", ESTIMATE_1E3["distance"], PAPER_FIGURES["code distance at 1e-3 (Table 2)"]),
    ratio_row("code distance, error rate 1e-6", ESTIMATE_1E6["distance"], PAPER_FIGURES["code distance at 1e-6 (Table 2)"]),
    ratio_row("physical runtime, error rate 1e-3 (days)", ESTIMATE_1E3["runtime_seconds"] / 86400, PAPER_FIGURES["runtime at 10 ns per T gate, days (Table 1)"]),
]))

,figure,paper,this reproduction,ours / paper
0,logical qubits,111,110,0.991
1,T gates,1.1e+15,1.14e+16,10.36
2,runtime at 10 ns per T gate (days),130,1319,10.15
3,"physical qubits, error rate 1e-3",1.8e+08,2.721e+06,0.01512
4,"physical qubits, error rate 1e-6",1.2e+06,1.815e+05,0.1513
5,"code distance, error rate 1e-3",35,39,1.114
6,"code distance, error rate 1e-6",9,9,1
7,"physical runtime, error rate 1e-3 (days)",130,1.64e+05,1261


## Machine-readable results for the QAC claim record

Everything the public record attributes to this reproduction is written here, from the live objects above: the
logical counts, both physical scenarios, the inputs they used, and the exact package versions. The cell is tagged
`results-export`; `qacclaims validate` checks that the committed `results.json` matches what this notebook
produces in its pinned environment.

In [14]:
CLAIM_ID = "femoco-reiher-2017"
REPRODUCTION_ID = "qdk-trotter-2026-11"


def package_version(name):
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return "unknown"


def scenario(estimate_, scenario_id):
    return {
        "scenario_id": scenario_id,
        "architecture": {
            "physical_error_rate": estimate_["physical_error_rate"],
            "gate_time_ns": PHYSICAL_GATE_TIME_NS,
            "measurement_time_ns": MEASUREMENT_TIME_NS,
        },
        "code": {
            "family": "surface",
            "distance": estimate_["distance"],
            "distance_source": "selected_by_tool",
            "cycle_time_ns": estimate_["cycle_time_ns"],
        },
        "factories": f"{estimate_['factory_copies']}×T",
        "synthesis": f"PSSPC with {estimate_['t_states_per_rotation']} T states per rotation; round-based 15-to-1 T factories; lattice surgery",
        "physical_qubits": estimate_["physical_qubits"],
        "compute_qubits": estimate_["compute_qubits"],
        "factory_qubits": estimate_["factory_qubits"],
        "memory_qubits": estimate_["memory_qubits"],
        "runtime_seconds": estimate_["runtime_seconds"],
        "error_budget": ERROR_BUDGET,
        "achieved_error": float(f"{estimate_['achieved_error']:.6g}"),
    }


results_record = {
    "schema_version": "1.0",
    "claim_id": CLAIM_ID,
    "reproduction_id": REPRODUCTION_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "producer": {"entrypoint": "notebook.ipynb", "cell_tag": "results-export"},
    "tooling": {
        "python": platform.python_version(),
        "packages": {name: package_version(name) for name in ("qdk-chemistry", "qdk", "numpy", "scipy")},
    },
    "inputs": {
        "hamiltonian": {
            "kind": "molecular_integrals",
            "orbitals": NUM_ORBITALS,
            "electrons": NUM_ELECTRONS,
            "source_url": INTEGRALS_URL,
            "source_sha256": INTEGRALS_SHA256,
            "pauli_coefficient_threshold": PAULI_COEFFICIENT_THRESHOLD,
        },
        "target_precision_hartree": TARGET_PRECISION,
        "trial_state": "not_modeled",
        "trotter": {
            "order": TROTTER_ORDER,
            "trotter_number": TROTTER_NUMBER,
            "trotter_number_reference_accuracy_hartree": TROTTER_NUMBER_REFERENCE_ACCURACY,
            "phase_estimation_constant": float(f"{PHASE_ESTIMATION_CONSTANT:.6g}"),
            "synthesis_t_per_rotation_slope": SYNTHESIS_SLOPE,
            "synthesis_t_per_rotation_offset": SYNTHESIS_OFFSET,
            "error_budget_rule": ERROR_BUDGET_RULE,
            "error_budget_phase_estimation_hartree": float(f"{MODEL['eps_phase']:.6g}"),
            "error_budget_trotter_hartree": float(f"{MODEL['eps_trotter']:.6g}"),
            "error_budget_synthesis_hartree": float(f"{MODEL['eps_synthesis']:.6g}"),
            "logical_t_gate_time_ns": LOGICAL_T_GATE_TIME_NS,
        },
    },
    "logical": {
        "walk_steps": TROTTER_STEPS,
        "num_qubits": LOGICAL_QUBITS,
        "t_count": 0,
        "rotation_count": ROTATION_COUNT,
        "rotation_depth": ROTATION_COUNT,
        "ccz_count": 0,
        "ccix_count": 0,
        "measurement_count": 0,
        "toffoli_count": 0,
        "pauli_terms": NUM_PAULI_TERMS,
        "rotations_per_step": ROTATIONS_PER_STEP,
        "t_states_per_rotation": T_PER_ROTATION,
        "t_count_after_synthesis": T_COUNT,
        "runtime_seconds_at_logical_t_rate": LOGICAL_RUNTIME_SECONDS,
    },
    "diagnostics": {
        "algorithm_qubits": ALGORITHM_QUBITS,
        "one_norm_hartree": float(f"{ONE_NORM:.6g}"),
        "pauli_terms_over_source_term_count": float(f"{NUM_PAULI_TERMS / PAPER_TERM_COUNT:.4g}"),
        "largest_validated_active_space_orbitals": max(VALIDATION_SIZES),
        "source_formula_trotter_steps": PAPER_MODEL["trotter_steps"],
        "source_formula_t_states_per_rotation": PAPER_MODEL["t_per_rotation"],
        "source_formula_t_count": PAPER_MODEL["t_count"],
        "source_formula_t_count_sum_rule": PAPER_MODEL_SUM["t_count"],
        "t_count_over_source_formula_t_count": float(f"{T_COUNT / PAPER_MODEL['t_count']:.4g}"),
        "logical_rate_runtime_years": float(f"{LOGICAL_RUNTIME_SECONDS / SECONDS_PER_YEAR:.4g}"),
        "runtime_years_error_rate_1e_3": float(f"{ESTIMATE_1E3['runtime_seconds'] / SECONDS_PER_YEAR:.4g}"),
        "runtime_years_error_rate_1e_6": float(f"{ESTIMATE_1E6['runtime_seconds'] / SECONDS_PER_YEAR:.4g}"),
        "t_factories_error_rate_1e_3": ESTIMATE_1E3["factory_copies"],
        "t_factories_error_rate_1e_6": ESTIMATE_1E6["factory_copies"],
        "t_states_per_rotation_error_rate_1e_3": ESTIMATE_1E3["t_states_per_rotation"],
        "t_states_per_rotation_error_rate_1e_6": ESTIMATE_1E6["t_states_per_rotation"],
        "logical_cycle_ns_error_rate_1e_3": ESTIMATE_1E3["cycle_time_ns"] * ESTIMATE_1E3["distance"],
    },
    "physical": [
        scenario(ESTIMATE_1E3, "surface-code-p1e-3"),
        scenario(ESTIMATE_1E6, "surface-code-p1e-6"),
    ],
}

with open("results.json", "w", encoding="utf-8", newline="\n") as handle:  # LF on every platform
    json.dump(results_record, handle, indent=2)
    handle.write("\n")

print(json.dumps(results_record["logical"], indent=2))
print({s["scenario_id"]: (s["physical_qubits"], round(s["runtime_seconds"])) for s in results_record["physical"]})

{
  "walk_steps": 6861089,
  "num_qubits": 110,
  "t_count": 0,
  "rotation_count": 172959950743891,
  "rotation_depth": 172959950743891,
  "ccz_count": 0,
  "ccix_count": 0,
  "measurement_count": 0,
  "toffoli_count": 0,
  "pauli_terms": 12604410,
  "rotations_per_step": 25208819,
  "t_states_per_rotation": 65.886,
  "t_count_after_synthesis": 11395639314712002,
  "runtime_seconds_at_logical_t_rate": 113956393.14712001
}
{'surface-code-p1e-3': (2721009, 14165419966), 'surface-code-p1e-6': (181529, 3268943069)}
